# The Physical Basis of SAR Ocean Measurements

## Motivation

Up to this point in the remote sensing sequence, most examples have used **passive sensors**. Passive sensors measure natural radiation, such as sunlight reflected by the ocean or thermal infrared emission from the sea surface.

Synthetic Aperture Radar (SAR) is different. SAR is an **active microwave sensor**: it transmits its own radar pulses toward Earth and measures the portion of that energy that is scattered back to the satellite. Because SAR supplies its own illumination and uses microwave wavelengths, it can observe the ocean during day or night and through most clouds.

For oceanography, SAR is especially useful because the radar backscatter is strongly affected by small-scale roughness at the ocean surface. Winds, waves, slicks, ice, ships, fronts, and current gradients can all change that roughness.

### Key Questions

Some key questions we might consider for SAR oceanography include:

- What makes SAR an active remote sensing system?
- How does a radar measure distance and image the surface?
- Why does microwave backscatter depend on ocean roughness?
- What ocean processes can be observed in SAR imagery?

In the sections below, we will use Python to build a few simple diagrams and toy calculations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Active vs Passive Remote Sensing

A passive sensor measures radiation that already exists. For example, an infrared SST sensor measures thermal radiation emitted by the ocean surface and modified by the atmosphere.

An active sensor carries both a transmitter and a receiver. A radar sends out a pulse of electromagnetic energy, waits for the echo, and records the returned signal. This gives radar several important advantages:

- It can operate without sunlight.
- Microwave wavelengths are much less affected by clouds than visible or infrared wavelengths.
- The measurement is sensitive to surface geometry, roughness, dielectric properties, and viewing angle.

The trade-off is that radar images are not usually as visually intuitive as optical images. They are images of **backscatter**, not color or temperature.

## How a Radar Measures Range

Radar systems determine distance using the travel time of a pulse. If a pulse takes time $\Delta t$ to travel from the satellite to the surface and back, the slant-range distance is:

$$R = rac{c\Delta t}{2}$$

where:

- $R$ is slant range from the radar to the target
- $c$ is the speed of light
- $\Delta t$ is the two-way travel time
- the factor of 2 accounts for the outgoing and returning path

In [ ]:
# Speed of light, m/s
c = 2.9979e8

# Example two-way travel times in milliseconds
delta_t_ms = np.array([4.0, 4.5, 5.0, 5.5, 6.0])
delta_t_s = delta_t_ms * 1e-3

# Slant range
R_km = (c * delta_t_s / 2) / 1000

for t, r in zip(delta_t_ms, R_km):
    print(f'Two-way time: {t:.1f} milliseconds  ->  slant range: {r:.0f} km')

## How SAR Improves Along-Track Resolution

A simple real-aperture radar has resolution that depends on antenna size and distance to the target. From orbit, obtaining fine along-track resolution with a real antenna alone would require an impractically large antenna.

SAR solves this by using the motion of the satellite. As the satellite flies forward, the same patch of ocean is observed over a short time from many slightly different positions. Processing those echoes together creates a **synthetic aperture** that is much longer than the physical antenna. This gives SAR its high spatial resolution.

## Microwave Bands Commonly Used by SAR

SAR instruments are often described by radar band. The band controls the wavelength, which affects surface interaction and penetration depth.

| Band | Approximate wavelength | Oceanographic notes |
|---|---:|---|
| X-band | ~3 cm | sensitive to very small capillary-gravity waves; often high resolution |
| C-band | ~5-6 cm | widely used for ocean winds, waves, sea ice, slicks, and ships |
| L-band | ~23-25 cm | longer wavelength; useful for some ice and coastal applications |
| S-band | ~9-12 cm | intermediate wavelength; used by some missions |

For most open-ocean SAR applications, the radar is not measuring the full wave height directly. Instead, it is often measuring changes in small-scale roughness that are linked to wind, waves, slicks, and currents.

In [ ]:
# Compare a few common SAR wavelengths
bands = ['X', 'C', 'S', 'L']
wavelength_cm = np.array([3.1, 5.6, 10.0, 24.0])
frequency_GHz = 29.979 / wavelength_cm  # c in cm/ns gives GHz approximately

plt.figure(figsize=(7, 4))
plt.bar(bands, wavelength_cm)
plt.ylabel('Wavelength (cm)')
plt.xlabel('Radar band')
plt.title('Approximate SAR Wavelengths')
plt.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.5)
plt.show()

for band, lam, freq in zip(bands, wavelength_cm, frequency_GHz):
    print(f'{band}-band: wavelength ~{lam:.1f} cm, frequency ~{freq:.2f} GHz')

## Backscatter and Surface Roughness

The basic SAR image value is related to **backscatter**: the fraction of transmitted energy scattered back toward the radar. In calibrated SAR products this is often represented as $\sigma^0$ and commonly shown in decibels:

$$\sigma^0_{dB} = 10 \log_{10}(\sigma^0)$$

Bright pixels usually indicate stronger backscatter. Dark pixels usually indicate weaker backscatter.

Over the ocean, C-band SAR backscatter is strongly influenced by centimeter-scale capillary-gravity waves. These small waves are generated by wind stress, so windier water is often brighter than calm water. However, interpretation is not one-to-one because viewing geometry, wave direction, slicks, rain cells, currents, ice, and ships can also modify the returned signal.

In [ ]:
# Convert between linear sigma0 and decibels
sigma0_linear = np.logspace(-4, -0.5, 100)
sigma0_db = 10 * np.log10(sigma0_linear)

plt.figure(figsize=(7, 4))
plt.semilogx(sigma0_linear, sigma0_db)
plt.xlabel('Linear backscatter coefficient')
plt.ylabel('Backscatter (dB)')
plt.title('Linear Backscatter Converted to Decibels')
plt.grid(linestyle='--', linewidth=0.5, alpha=0.5)
plt.show()

print(f'A linear backscatter of 0.01 is {10*np.log10(0.01):.1f} dB.')
print(f'A linear backscatter of 0.10 is {10*np.log10(0.10):.1f} dB.')

## Incidence Angle Matters

The **incidence angle** is the angle between the incoming radar beam and the local vertical direction at the surface. Backscatter from the ocean usually changes across a SAR swath because incidence angle changes from near range to far range.

This is one reason SAR images often require calibration and normalization before quantitative interpretation. A dark region might be a real ocean feature, or it might be partly related to viewing geometry.

In [ ]:
# A toy incidence-angle dependence for ocean backscatter.
# This is not a retrieval algorithm; it is only for illustrating geometry effects.
incidence = np.linspace(20, 45, 200)
sigma0_db_toy = -7 - 0.25 * (incidence - 20)

plt.figure(figsize=(7, 4))
plt.plot(incidence, sigma0_db_toy)
plt.xlabel('Incidence angle (degrees)')
plt.ylabel('Toy ocean backscatter (dB)')
plt.title('Backscatter Often Changes Across a SAR Swath')
plt.grid(linestyle='--', linewidth=0.5, alpha=0.5)
plt.show()

## What SAR Sees on the Ocean

SAR does not directly measure temperature, chlorophyll, or sea surface height. It images microwave backscatter from the surface. Oceanographic interpretation comes from understanding how processes alter surface roughness or introduce strong targets.

Examples include:

- **Wind speed patterns:** higher wind stress usually increases small-scale roughness and backscatter.
- **Surface slicks and oil:** films can damp capillary waves, making dark features.
- **Internal waves and current fronts:** convergence and divergence patterns modulate surface roughness.
- **Ocean swell:** long waves can appear through imaging mechanisms that depend on wave direction and motion.
- **Sea ice:** ice type, roughness, wetness, and deformation strongly affect radar backscatter.
- **Ships:** metal structures often produce very bright point targets.

## Summary

SAR oceanography begins with a different measurement philosophy than passive ocean color or infrared SST:

1. SAR is an active microwave system that transmits pulses and records echoes.
2. Range is determined from two-way travel time.
3. High along-track resolution comes from synthetic aperture processing.
4. Ocean SAR brightness is related to backscatter, especially from wind-roughened small waves.
5. Interpretation requires attention to incidence angle, polarization, wavelength, speckle, and ocean conditions.

# Check Your Understanding

1. Why can SAR observe the ocean at night while visible ocean color sensors cannot?
2. A radar echo returns after 5 milliseconds. What is the approximate slant range?
3. Why does SAR use a synthetic aperture instead of relying only on a physical antenna?
4. What is one ocean process that can make SAR backscatter brighter? What is one process that can make it darker?
5. Why must incidence angle be considered when interpreting a SAR image?

## References and Further Reading

- NASA Earthdata/ARSET, *An Introduction to Synthetic Aperture Radar and Its Applications*.
- NASA JPL, *Synthetic Aperture Radars Imaging Basics*.
- ESA Sentinel-1, *Oceans and ice*.
- NOAA CoastWatch, *Monitoring Sea Surface Winds and Sea Ice with Satellite Radar*.
- SAR Marine User's Manual.